## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: K-Means

***

In [5]:
# API UCI Repository
# #
!pip install ucimlrepo

In [6]:
## Librerias
from pandas import DataFrame
from tqdm import tqdm

import seaborn as sns
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from sklearn.decomposition import PCA

## Carga de datos

__Dataset:__

Estos datos son referenciados en [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/109/wine) que son los resultados de un análisis químico de vinos cultivados en una misma región de Italia pero derivados de cultivos diferentes. 

<center>
    <img src='https://media.licdn.com/dms/image/v2/D5612AQHwzFJW75X22A/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1674384291032?e=2147483647&v=beta&t=IBguDw6af4l1PyaTlV6Rw1YilLGFbUdn2_y178PYdww' width=800>
</center>

In [ ]:
## Cargar objeto de datos
dataset = fetch_ucirepo(id=109)

## Extraer los features del dataset
data = dataset.data.features 
print('(shape) data: {}'.format(data.shape))

## Mostrar los primeros 5 registros
data.head()

In [ ]:
data.describe()

In [ ]:
#print(dataset['metadata']['additional_info']['summary'])

#### Escalamos los datos

In [ ]:
## Instancia del modelo y ajuste
scaler = StandardScaler().fit(data)

## Aplicación del escalado a los datos
data_scaled = scaler.transform(data)
data_scaled

In [ ]:
## Convertir a un dataframe
data_scaled = DataFrame(data=data_scaled,
                        columns=data.columns)
data_scaled.head()

#### Veamos algunas gráficas preliminares

In [ ]:
## Grafica de dispersión
plt.figure(figsize=(10, 8))
sns.scatterplot(data=data_scaled, 
                x="Alcohol", 
                y="Color_intensity")
plt.show()

In [ ]:
## Grafica de dispersión
plt.figure(figsize=(10, 8))
sns.scatterplot(data=data_scaled, 
                x="Malicacid", 
                y="Total_phenols")
plt.show()

In [ ]:
## Duración: 35seg aprox en colab

## Grafica pairplot
plt.figure(figsize=(10, 10))
sns.pairplot(data=data_scaled)
plt.show()

## Clase k-means

Existen múltiples hiperparámetros para el modelo k-means:<br>

```{python}
    km_model = KMeans(n_clusters=8, init='k-means++', 
                      n_init=10, max_iter=300, 
                      tol=0.0001, random_state=None)
```

| Hiperparámetros | Descripción |
|-----------------|-------------|
| __n_clusters__  | número de clusters.|
| __init__ | 'k-means++' inicialización inteligente, 'random' aleatoria. |
| __n_init__ | Número de veces que aplicaremos k-means.| 
| __max_iter__ | Máximo número de iteraciones para cada ejecución.|
| __tol__ | Tolerancia para la convergencia.|
| __random_state__ | Semilla para inicializar los centroides. Use un entero para ser determinista.|

Existen múltiple métodos/funciones para el modelo k-means

* __km_model.fit(data)__ : entrena el modelo usando ciertos datos.

* __km_model.predict(data)__: dado un modelo entrenado, determina a que clase pertenece cada punto retornando un vector con predicciones.

In [ ]:
## Numero de clusters a definir
nro_clusters = 4

## Instancia del modelo
km_model = KMeans(n_clusters=nro_clusters, 
                  n_init=100, random_state=9001)

## Ajuste del modelo con los datos
km_model.fit(data_scaled)

## Caracteristicas del modelo entrenado
Una vez entrenado el modelo, existen nuevas características que podemos observar (atributos)<br>

| Atributos | Descripción |
|-----------|-------------|
| cluster_centers_ | Las coordenadas de los centroides. Si el algoritmo no converge estos no serán consistentes con los labels.|
| inertia_ | La suma total de los distancias intra-cluster. |
| labels_ | Las etiquetas de cada punto (cluster al que pertenece). |
| n_iter_ | Número de iteraciones del algoritmo. |

In [ ]:
## Mostrar los centros de los clusters
print('Centros: \n {}\n'.format(km_model.cluster_centers_))

## Distancia intra-cluster
print('Distancia intra-cluster: {:.4f}\n'.format(km_model.inertia_))

## Etiquetas de las muestras usadas para entrenar
print('Etiquetas: \n {}\n'.format(km_model.labels_))

## Numero de iteraciones realizadas
print('Numero de iteraciones: {}'.format(km_model.n_iter_))

## Graficando los clusters

In [ ]:
## Creamos un dataframe con los datos y la etiqueta asignada
data_temp = data_scaled.copy()
data_temp['class'] = km_model.labels_
data_temp

In [ ]:
sns.color_palette()

In [ ]:
nro_clusters

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=data_temp, 
                x="Alcohol", 
                y="Color_intensity",
                hue='class', 
                palette=sns.color_palette()[:nro_clusters])
plt.legend(loc=[1.01, 0.5])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=data_temp, 
                x="Malicacid", 
                y="Total_phenols",
                hue='class', 
                palette=sns.color_palette()[:nro_clusters])
plt.legend(loc=[1.01, 0.5])
plt.tight_layout()
plt.show()

## Buscando el valor de K

In [ ]:
## Maxima cantidad de clusteres
max_K = 20

## Almacenado de SSE-score
sse = []

for k in tqdm(list(range(1, max_K+1))):

    ## Instancia del modelo
    km_model = KMeans(n_clusters=k, n_init=100, random_state=9001)

    ## Ajuste del modelo
    km_model.fit(data_scaled)

    ## Guardar el SSE obtenido
    sse.append(km_model.inertia_)


In [ ]:
sse

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, max_K+1), sse)
plt.title('Elbow curve')
plt.xlabel('Numero de cluster')
plt.ylabel('SSE')
plt.tight_layout()
plt.show()

#### Mejor modelo

In [ ]:
## Ajuste del mejor modelo

## Numero de clusters a definir
nro_clusters = 3

## Instancia del modelo
km_model = KMeans(n_clusters=nro_clusters, 
                  n_init=100, random_state=9001)

## Ajuste del modelo con los datos
km_model.fit(data_scaled)

In [ ]:
km_model.labels_

In [ ]:
km_model.predict(data_scaled)

## Interpretación de los clústeres
#### PCA

In [ ]:
#Creando el objeto y aplicando PCA
pca_model = PCA(n_components=2)
pca_model.fit(data_scaled)
pca_Data = pca_model.transform(data_scaled)

## Crear un dataframe con los datos aplicado a PCA y las etiquetas generada por K-Means
pca_Data = DataFrame(pca_Data,columns=["PC1","PC2"])
pca_Data["labels"]=km_model.labels_
pca_Data

In [ ]:
## Graficado de PCA
plt.figure(figsize=(7, 5))
sns.scatterplot(data=pca_Data, 
                x='PC1', y='PC2', 
                hue='labels',
                palette=sns.color_palette()[:nro_clusters])
plt.legend(loc=[1.01, 0.5], title='Label')
plt.tight_layout()
plt.show()

#### Analizando los componentes

In [ ]:
pca_components = DataFrame(data=pca_model.components_.T,
                           columns=['PC1', 'PC2'])
pca_components['feature'] = data_scaled.columns
pca_components

In [ ]:
plt.figure(figsize=(7, 5))
for _, row in pca_components.iterrows():
    plt.scatter(row['PC1'], row['PC2'], s=100)
    plt.arrow(
        x=0, y=0, # coordinates of arrow base
        dx=row['PC1'], # length of the arrow along x
        dy=row['PC2'], # length of the arrow along y
        color='r', 
        head_width=0.01
        )
    plt.text(row['PC1'], row['PC2'], row['feature'])
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

In [ ]:
## Facotr de escalado de los datos transformados (solo usado para esta grafica)
scalePC1 = 1.0/(pca_Data['PC1'].max() - pca_Data['PC1'].min())
scalePC2 = 1.0/(pca_Data['PC2'].max() - pca_Data['PC2'].min())

## Graficador
plt.figure(figsize=(10, 8))
sns.scatterplot(x=pca_Data['PC1']*scalePC1, 
                y=pca_Data['PC2']*scalePC2, 
                hue=pca_Data['labels'],
                palette=sns.color_palette()[:nro_clusters])

for _, row in pca_components.iterrows():
    plt.scatter(row['PC1'], row['PC2'], s=100)
    plt.arrow(
        x=0, y=0, # coordinates of arrow base
        dx=row['PC1'], # length of the arrow along x
        dy=row['PC2'], # length of the arrow along y
        color='r', 
        head_width=0.01
        )
    plt.text(row['PC1'], row['PC2'], row['feature'])
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Biplot')
plt.grid()
plt.tight_layout()
plt.show()

Caracterización de los clústeres:

* Cluster 0: Alto en Flavanoids y Total phenols y Proline, intermedio en Proanthonyanins.
* Cluster 1: Alto en Alcalinity of ash, nonflavanoid phenols y Malicacid.
* Cluster 2: Bajo en Ash, Color intensity, Alcohol y Magnesium.

#### Analizando los centroides

In [ ]:
## Se extrae los centroides
km_centroids = DataFrame(data=km_model.cluster_centers_, 
                         columns=data_scaled.columns)
km_centroids

In [ ]:
## Se calcula la desviación del centroide de cada variable asociada
km_centroids_summary = km_centroids.std().sort_values(ascending=True).reset_index()
km_centroids_summary.columns = ['var_names', 'std']
km_centroids_summary

In [ ]:
## Gráficación de la desviación estandar de los centroides
km_centroids_summary.plot(kind='barh', legend=None, figsize=(8, 4))
plt.yticks(ticks=km_centroids_summary.index, 
           labels=km_centroids_summary['var_names'])
plt.xlabel('desviacion estandar')
plt.ylabel('Variables')
plt.title('Desviacion estandar de los centroides')
plt.show()

#### Revisado de desviaciones por grupo generado

In [ ]:
## Crear una tabla con data real agregando las etiquetas generada del clustering
data_cluster = data.copy()
data_cluster['cluster'] = km_model.labels_
data_cluster

In [ ]:
## Se reestructura los datos para efecto de visualizacion
data_cluster_melt = data_cluster.melt(id_vars='cluster')
data_cluster_melt

In [ ]:
## Mostrar para una de las variables la distribución de cada grupo
var = "Alcohol"
temp = data_cluster_melt.query(f'variable == "{var}"')
sns.pointplot(
    data=temp, x="cluster", y="value",
    errorbar=('pi', 100), capsize=.4, join=False, color="blue",
)
plt.title(f'Var: {var}')

In [ ]:
## Graficado de la distribución de cada variable por grupo

ncol = 5
nrow = data_cluster.shape[1] // ncol +1
nrow = nrow if nrow>0 else 1
print('nrow: {} - ncol: {}'.format(nrow, ncol))

plt.figure(figsize=(16, 4*nrow))
for i, var in enumerate(data.columns):

    plt.subplot(nrow, ncol, i+1)
    temp = data_cluster_melt.query(f'variable == "{var}"')
    sns.pointplot(
        data=temp, x="cluster", y="value",
        errorbar=('pi', 100), capsize=.4, join=False, color="blue",
    )
    plt.title(f'Var: {var}')
plt.tight_layout()
plt.show()